# Word Algebra: word2vec vector arithmetic

`king - man + woman` lands close to `queen`. Word embeddings turn words into points
in space, and the *directions* between them turn out to mean things — plural,
gender, capital-of-a-country, comparative-of-an-adjective — well enough that you
can add and subtract your way from one word to another.

This notebook is the Python/gensim companion to the **Word Algebra** browser app.
Both draw on the same [GloVe](https://nlp.stanford.edu/projects/glove/) vectors
(Pennington, Socher & Manning, EMNLP 2014, Stanford NLP, 100 dimensions) — but this
notebook loads the **full 400,000-word vocabulary**, while the browser app only
ships its most frequent 100,000 words. So a rare word might resolve here but come
back "not in vocabulary" in the browser tool.

**To use it:** `Runtime > Run all`, then scroll to the equation box near the bottom
and type your own `word - word + word` expressions. No coding required.

## 1. Install & load the vectors

The first run downloads ~130MB of vectors and can take a minute or two.

In [ ]:
!pip install -q gensim ipywidgets


In [ ]:
import gensim.downloader as api

print("Loading GloVe (100d, 400,000 words)... this can take a minute the first time.")
model = api.load("glove-wiki-gigaword-100")
print(f"Loaded {len(model.key_to_index):,} words, {model.vector_size} dimensions.")


## 2. The equation parser

Same syntax as the browser app: type a word, then `+` or `-` before the next one.
The first word is always added.

In [ ]:
import re

def parse_equation(text):
    """'king - man + woman' -> [('king', 1), ('man', -1), ('woman', 1)]"""
    tokens = re.findall(r"[+-]?\s*[a-zA-Z']+", text.strip().lower())
    terms = []
    for tok in tokens:
        tok = tok.strip()
        sign = 1
        if tok.startswith("-"):
            sign, tok = -1, tok[1:].strip()
        elif tok.startswith("+"):
            tok = tok[1:].strip()
        if tok:
            terms.append((tok, sign))
    return terms


## 3. Try an equation

Computes the top-5 nearest words (gensim's `most_similar`, the same "3CosAdd"
analogy math the browser app mirrors) and plots a 2D map (PCA) of the words
involved: the running total as each term is applied, and where the top matches
land relative to it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PLUS, MINUS, RESULT, PATH = "#2a78d6", "#eb6834", "#1baf7a", "#8b969c"

def run_equation(text):
    terms = parse_equation(text)
    if not terms:
        print("Type a word to get started.")
        return

    missing = [w for w, _ in terms if w not in model.key_to_index]
    if missing:
        print(f"Not in the 400,000-word vocabulary: {', '.join(missing)}")
        return

    positive = [w for w, s in terms if s > 0]
    negative = [w for w, s in terms if s < 0]
    results = model.most_similar(positive=positive, negative=negative, topn=5)

    print(f"{text} =")
    for word, score in results:
        print(f"  {word:<20s} {score:.3f}")

    # running cumulative path, unit vectors at each step (mirrors the browser app)
    path_labels, path_vecs = ["start"], [np.zeros(model.vector_size)]
    running = np.zeros(model.vector_size)
    for w, s in terms:
        running = running + s * (model[w] / np.linalg.norm(model[w]))
        path_vecs.append(running.copy())
        path_labels.append(("+" if s > 0 else "−") + w)

    result_words = [w for w, _ in results]
    result_vecs = [model[w] / np.linalg.norm(model[w]) for w in result_words]

    all_vecs = np.array(path_vecs + result_vecs)
    centered = all_vecs - all_vecs.mean(axis=0)
    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    proj = centered @ Vt[:2].T
    path_proj, result_proj = proj[:len(path_vecs)], proj[len(path_vecs):]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axhline(0, color="#dde3e3", linewidth=1, zorder=0)
    ax.axvline(0, color="#dde3e3", linewidth=1, zorder=0)
    ax.plot(path_proj[:, 0], path_proj[:, 1], "--", color=PATH, linewidth=1, zorder=1)
    ax.scatter(*path_proj[0], color=PATH, s=30, zorder=2)
    ax.annotate("start", path_proj[0], textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9, color=PATH)
    for i, (x, y) in enumerate(path_proj[1:]):
        is_last = i == len(path_proj) - 2
        ax.scatter(x, y, color=PLUS if terms[i][1] > 0 else MINUS,
                   s=70 if is_last else 50, zorder=2,
                   edgecolors="black" if is_last else "none", linewidths=1)
        ax.annotate(path_labels[i + 1], (x, y), textcoords="offset points",
                    xytext=(0, 9 if i % 2 == 0 else -15), ha="center",
                    fontsize=9, fontweight="bold")
    for i, (x, y) in enumerate(result_proj):
        ax.scatter(x, y, color=RESULT, s=90 - i * 10, alpha=0.85, zorder=2)
        ax.annotate(result_words[i], (x, y), textcoords="offset points",
                    xytext=(0, 9 if i % 2 == 0 else -15), ha="center",
                    fontsize=9, fontweight="bold", color="#0f7d54")
    ax.set_title(text, fontfamily="monospace")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

equation_box = widgets.Text(value="king - man + woman", description="Equation:",
                             layout=widgets.Layout(width="420px"))
compute_btn = widgets.Button(description="Compute", button_style="primary")
output = widgets.Output()

def on_compute(_):
    with output:
        clear_output()
        run_equation(equation_box.value)

compute_btn.on_click(on_compute)
equation_box.on_submit(on_compute)
display(widgets.HBox([equation_box, compute_btn]), output)
on_compute(None)


## 4. What did we learn?

Word vectors capture real semantic relationships that let vector arithmetic
"complete an analogy" — but they also absorb whatever patterns sit in the text
they were trained on. Try `doctor - man + woman`: many word2vec/GloVe models
answer *nurse* — not because doctors can't be women, but because that's the
pattern in the decades of news and web text these vectors learned from. That's
worth discussing openly, not something to hide.

Other things worth trying in class:
- `walked - walk + swim` (verb tense) and `tallest - tall + fast` (superlatives)
- A rare or invented word (e.g. `supercalifragilisticexpialidocious`) — it won't
  be in the vocabulary here either, since GloVe only knows words that appeared
  often enough in its training corpus
- Comparing an equation's result here against the same equation in the browser
  app — they should mostly agree, since both use the same 100-dimensional GloVe
  space, just with different vocabulary sizes (400,000 here vs. 100,000 there)

**Source:** GloVe 6B (Pennington, Socher & Manning, EMNLP 2014), Stanford NLP,
Public Domain (PDDL). https://nlp.stanford.edu/projects/glove/